# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector Construction Plan

For this exercise, we will create a synthetic dataset to demonstrate feature engineering. We will focus on:
1.  **Creating a dummy DataFrame.**
2.  **Handling missing values.**
3.  **Creating a new feature.**

In [1]:
import pandas as pd
import numpy as np

# Create a dummy DataFrame
data = {
    'feature_A': np.random.rand(100),
    'feature_B': np.random.randint(1, 10, 100),
    'categorical_C': np.random.choice(['X', 'Y', 'Z'], 100),
    'target': np.random.randint(0, 2, 100)
}
df = pd.DataFrame(data)

# Introduce some missing values for demonstration
df.loc[df.sample(frac=0.1).index, 'feature_A'] = np.nan
df.loc[df.sample(frac=0.05).index, 'categorical_C'] = np.nan

print("Original DataFrame head:")
display(df.head())
print("\nMissing values before handling:")
display(df.isnull().sum())

Original DataFrame head:


,feature_A,feature_B,categorical_C,target
0,0.378453,2,X,1
1,0.101088,3,X,0
2,0.884299,4,X,1
3,0.735922,1,X,1
4,0.904027,7,X,1



Missing values before handling:


,0
feature_A,10
feature_B,0
categorical_C,5
target,0


In [2]:
# 1. Handling missing values:
# For numerical 'feature_A', fill with the mean.
df['feature_A'] = df['feature_A'].fillna(df['feature_A'].mean())

# For categorical 'categorical_C', fill with the mode.
df['categorical_C'] = df['categorical_C'].fillna(df['categorical_C'].mode()[0])

# 2. Creating a new feature:
# 'interaction_AB' as product of feature_A and feature_B
df['interaction_AB'] = df['feature_A'] * df['feature_B']

# 3. One-hot encode the categorical feature
df = pd.get_dummies(df, columns=['categorical_C'], prefix='cat', drop_first=True)

print("\nDataFrame head after feature engineering:")
display(df.head())
print("\nMissing values after handling:")
display(df.isnull().sum())


DataFrame head after feature engineering:


,feature_A,feature_B,target,interaction_AB,cat_Y,cat_Z
0,0.378453,2,1,0.756906,False,False
1,0.101088,3,0,0.303263,False,False
2,0.884299,4,1,3.537198,False,False
3,0.735922,1,1,0.735922,False,False
4,0.904027,7,1,6.328192,False,False



Missing values after handling:


,0
feature_A,0
feature_B,0
target,0
interaction_AB,0
cat_Y,0
cat_Z,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

Here's a brief description of the features in the final `df`:

*   **feature_A**: Original numerical feature. Missing values were imputed with the mean.
*   **feature_B**: Original numerical feature, no missing values.
*   **target**: The binary target variable, no missing values.
*   **interaction_AB**: A new numerical feature created by multiplying `feature_A` and `feature_B`.
*   **cat_Y**: A binary indicator (0 or 1) for the 'Y' category from one-hot encoding of `categorical_C`. Missing values in `categorical_C` were imputed with the mode before encoding.
*   **cat_Z**: A binary indicator (0 or 1) for the 'Z' category from one-hot encoding of `categorical_C`. Missing values in `categorical_C` were imputed with the mode before encoding.

All features are available before the moment of prediction as they are derived from the initial dataset without future information.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Excluded Features

In this demonstration, we intentionally created a `leaky_feature` to illustrate data leakage. In a real-world scenario, any feature that is directly derived from the target variable or incorporates information that would not be available at the time of prediction must be excluded to prevent artificially inflated model performance. Therefore, the `leaky_feature` would be excluded from the final feature set for model training.

**Excluded Feature:**
*   `leaky_feature`: Excluded because it is directly derived from the `target` variable, making it a source of data leakage.

### Leakage Hunt

Data leakage occurs when information from outside the training data is used to create the model, leading to overly optimistic performance estimates. This can happen if features are derived from the target variable or include future information not available at prediction time.

Here, we'll demonstrate a hypothetical leakage scenario by creating a feature (`leaky_feature`) that is directly derived from the `target` variable. In a real-world scenario, such a feature would lead to an artificially high model performance because the model would effectively be 'seeing' the answer.

In [3]:
# Demonstrate a hypothetical leakage scenario:
# Create a 'leaky_feature' highly correlated with the target
df['leaky_feature'] = df['target'] * 0.9 + np.random.rand(len(df)) * 0.1

print("DataFrame head with leaky feature:")
display(df.head())

print("Correlation of leaky_feature with target:")
display(df[['target', 'leaky_feature']].corr())

# Note: In a real scenario, 'leaky_feature' would be excluded
# if it were derived from the target or future information.

DataFrame head with leaky feature:


,feature_A,feature_B,target,interaction_AB,cat_Y,cat_Z,leaky_feature
0,0.378453,2,1,0.756906,False,False,0.912827
1,0.101088,3,0,0.303263,False,False,0.094465
2,0.884299,4,1,3.537198,False,False,0.939466
3,0.735922,1,1,0.735922,False,False,0.919493
4,0.904027,7,1,6.328192,False,False,0.939967


Correlation of leaky_feature with target:


,target,leaky_feature
target,1.000000,0.998043
leaky_feature,0.998043,1.000000


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [4]:
print("DataFrame columns before excluding leaky_feature:")
display(df.columns)

# Exclude the 'leaky_feature' from the DataFrame
df = df.drop(columns=['leaky_feature'])

print("\nDataFrame columns after excluding leaky_feature:")
display(df.columns)

print("\nFinal DataFrame head without leaky_feature:")
display(df.head())

DataFrame columns before excluding leaky_feature:


Index(['feature_A', 'feature_B', 'target', 'interaction_AB', 'cat_Y', 'cat_Z',
       'leaky_feature'],
      dtype='object')


DataFrame columns after excluding leaky_feature:


Index(['feature_A', 'feature_B', 'target', 'interaction_AB', 'cat_Y', 'cat_Z'], dtype='object')


Final DataFrame head without leaky_feature:


,feature_A,feature_B,target,interaction_AB,cat_Y,cat_Z
0,0.378453,2,1,0.756906,False,False
1,0.101088,3,0,0.303263,False,False
2,0.884299,4,1,3.537198,False,False
3,0.735922,1,1,0.735922,False,False
4,0.904027,7,1,6.328192,False,False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.